In [ ]:
import os
import shutil
import random
from tqdm import tqdm
import yaml
from PIL import Image 

raw_data_path = "/home_data/hejx/archive/data"
output_path = "pre_dataset"
#类别映射
class_mapping = {
    "c0": (0, "normal_driving"),       #正常驾驶
    "c1": (1, "right_hand_messaging"), #右手发消息
    "c2": (2, "right_hand_calling"),   #右手打电话
    "c3": (3, "left_hand_messaging"),  #左手发消息
    "c4": (4, "left_hand_calling"),    #左手打电话
    "c5": (5, "adjusting_radio"),      #调整收音机
    "c6": (6, "drinking_water"),       #喝水
    "c7": (7, "holding_objects"),      #拿东西
    "c8": (8, "adjusting_clothing"),   #整理着装
    "c9": (9, "talking_to_passenger")  #与乘客交流
}
# 划分训练集/验证集
train_ratio = 0.8
SUPPORTED_FORMATS = (".jpg", ".jpeg", ".png", ".bmp")
def create_folder_structure():#创建文件夹
    dirs = [
        os.path.join(output_path, "images/train"),
        os.path.join(output_path, "images/val"),
        os.path.join(output_path, "labels/train"),
        os.path.join(output_path, "labels/val")
    ]
    for dir_path in dirs:
        os.makedirs(dir_path, exist_ok=True)
    print(f"已创建文件夹结构：{output_path}")

def process_dataset():
    create_folder_structure()
    for class_name, class_id in tqdm(class_mapping.items(), desc="处理所有类别"):
        class_raw_path = os.path.join(raw_data_path, class_name)
        if not os.path.exists(class_raw_path):
            print(f"类别文件夹不存在，跳过：{class_raw_path}")
            continue
        # 获取当前类别下的所有图像文件
        image_files = [
            f for f in os.listdir(class_raw_path) 
            if f.lower().endswith(SUPPORTED_FORMATS)
        ]
        if not image_files:
            print(f"类别{class_name}下无图像文件，跳过")
            continue
        #随机打乱数据（保证划分的随机性）
        random.shuffle(image_files)
        split_idx = int(len(image_files) * train_ratio)
        train_files = image_files[:split_idx]
        val_files = image_files[split_idx:]
        
        process_files(train_files, class_raw_path, class_id, "train")#处理训练集
        process_files(val_files, class_raw_path, class_id, "val")#处理验证集
def process_files(files, class_raw_path, class_id, split_type):
    for img_file in files:
        #复制图像到目标目录
        src_img = os.path.join(class_raw_path, img_file)
        dst_img = os.path.join(output_path, f"images/{split_type}", img_file)
        shutil.copy(src_img, dst_img)
        
        #生成YOLO格式标签（关键：读取图像真实尺寸）
        img_name = os.path.splitext(img_file)[0]
        label_file = f"{img_name}.txt"
        dst_label = os.path.join(output_path, f"labels/{split_type}", label_file)
        
        #读取图像真实宽高（避免硬编码尺寸）
        try:
            with Image.open(src_img) as img:
                img_width, img_height = img.size
        except Exception as e:
            print(f"读取图像{img_file}尺寸失败，使用默认640x480：{e}")
            img_width, img_height = 640, 480
        
        #YOLO标签格式：类别ID 中心点x 中心点y 宽度 高度（相对坐标）
        center_x = 0.5  #中心点x（相对宽）
        center_y = 0.5  #中心点y（相对高）
        width = 1.0     #目标宽度（相对宽）
        height = 1.0    #目标高度（相对高）
        
        #写入标签文件
        with open(dst_label, "w", encoding="utf-8") as f:
            f.write(f"{class_id} {center_x:.6f} {center_y:.6f} {width:.6f} {height:.6f}")
def generate_yaml():
    yaml_data = {
        "train": os.path.join(os.path.abspath(output_path), "images/train"),
        "val": os.path.join(os.path.abspath(output_path), "images/val"), 
        "nc": len(class_mapping),  #类别数
        "names": [semantic_name for (_, semantic_name) in class_mapping.values()]  #类别名称
    }
    yaml_path = os.path.join(output_path, "driving_data.yaml")
    with open(yaml_path, "w", encoding="utf-8") as f:
        yaml.dump(yaml_data, f, indent=2)
    
    print(f"YAML配置文件已生成：{yaml_path}")
    # 打印YAML内容（方便检查）
    print("\nYAML配置内容：")
    for k, v in yaml_data.items():
        print(f"{k}:{v}")
        
if __name__ == "__main__":
    random.seed(42)#固定随机种子（保证每次划分结果一致）
    process_dataset()#执行预处理
    generate_yaml()#生成YAML文件
    train_img_num = len(os.listdir(os.path.join(output_path, "images/train")))
    val_img_num = len(os.listdir(os.path.join(output_path, "images/val")))
    print(f"\n数据集预处理完成！")
    print(f"  训练集图像数量：{train_img_num}")
    print(f"  验证集图像数量：{val_img_num}")
    print(f"  总类别数：{len(class_mapping)}")
    print(f"  预处理结果路径：{os.path.abspath(output_path)}")